# Baseline v3 — Retrieval exploration guided by ragas

**Goal.** Build the best possible retrieval pipeline for CUAD contract clauses *before* touching generation, then choose a generation model on top of it.

**Why this notebook exists.** In baseline v2 the exploration was guided by `hit_rate` (does the reference text appear in the retrieved chunks?). It only measures recall and ignores noise and ranking: `hit_rate` improved by about 50% while ragas `context_precision` actually dropped (0.409 → 0.366). Baseline v3 therefore redoes the exploration with ragas `context_precision` and `context_recall` from the very first step.

**How results stay comparable.**
- The same fixed sample is used everywhere: 205 positive examples (5 per category, 41 categories) and 118 negative examples (clause absent from the contract).
- Retrieval is always scoped to a single contract (metadata filter on the contract title).
- `context_precision` and `context_recall` only need the question, the retrieved chunks and the reference answer. **No generated answer is required**, so each retrieval configuration can be scored without running a local LLM.

**Roadmap.**
1. Embedding models
2. Hybrid search (dense + BM25)
3. Reranking and positional retrieval for `Document Name` / `Parties` (Tier 1)
4. Adaptive `top_k` for hard categories (Tier 3)
5. Generation: model and prompt, then final decision


## Step 0 — Setup: reuse what baseline v2 already built

Nothing heavy is rebuilt here. The cells below reload:
- the fixed evaluation sample from the baseline v2 checkpoint (`positive_sample`, 205 examples);
- the local Qdrant collections built in v2 (one per embedding model: bge-small, bge-large, nomic);
- the three embedding models, and a simple scoped retriever (`retrieve_scoped`): dense search, `top_k = 3`, restricted to the chunks of a single contract.

This plain retriever is the starting point of the exploration: no hybrid search, no reranking, no per-category tuning yet.


In [1]:
import pickle
import pandas as pd
from qdrant_client import QdrantClient
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.ollama import OllamaEmbedding

/Users/aitbellalhamza/Desktop/RAG-for-contract-understanding-CUAD/.cuad-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("../artifacts/baseline_v2_checkpoint.pkl", "rb") as f:
    checkpoint_v2 = pickle.load(f)

positive_sample = checkpoint_v2["positive_sample"]
len(positive_sample)

205

In [3]:
qdrant_client = QdrantClient(path="../artifacts/qdrant_storage")

/var/folders/2d/1h410ps12gn5kw05_3p23qpc0000gn/T/ipykernel_11022/4267565823.py:1: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <cuad_sentence_widow_hybrid> contains 50675 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant_client = QdrantClient(path="../artifacts/qdrant_storage")
/var/folders/2d/1h410ps12gn5kw05_3p23qpc0000gn/T/ipykernel_11022/4267565823.py:1: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <cuad_sentence_window6_hybrid> contains 50675 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant_client = QdrantClient(path="../artifacts/qdrant_storage")


In [5]:
bge_small_embed = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
bge_large_embed = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5")
nomic_embed = OllamaEmbedding(model_name="nomic-embed-text")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4791.25it/s]


In [6]:
def load_index(collection_name, embed_model):
    vector_store = QdrantVectorStore(
        client=qdrant_client,
        collection_name=collection_name
    )
    return VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)

index_bge_small = load_index("cuad_bge_small", bge_small_embed)
index_bge_large = load_index("cuad_bge_large", bge_large_embed)
index_nomic = load_index("cuad_nomic", nomic_embed)

In [7]:
def retrieve_scoped(index, question, title, top_k=3):
    filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    retriever = index.as_retriever(similarity_top_k=top_k, filters=filters)
    nodes = retriever.retrieve(question)
    return [n.node.get_content() for n in nodes]

## Evaluation setup — ragas with an LLM judge

Retrieval is scored with two ragas metrics, judged by Claude Haiku 4.5:
- **`context_precision`**: are the retrieved chunks relevant to the reference answer, and well ranked?
- **`context_recall`**: do the retrieved chunks cover the reference answer?

Two implementation notes:
- Recent Claude models reject the `temperature` parameter, but ragas forces it. `ChatAnthropicNoTemp` silently ignores that assignment, and `max_tokens=4096` avoids truncated judge outputs. These two metrics do not need an embedding model, so none is passed to ragas here.
- Because the temperature is not pinned, the judge is slightly non-deterministic. Re-scoring identical chunks moves global scores by about ±0.03, and per-category scores (5 examples each) by up to ±0.1–0.2. Small differences should not be over-interpreted.


In [8]:
from ragas.metrics import context_precision, context_recall
from ragas.run_config import RunConfig
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LlamaIndexEmbeddingsWrapper

class ChatAnthropicNoTemp(ChatAnthropic):
    def __setattr__(self, name, value):
        if name == "temperature":
            return
        super().__setattr__(name, value)

claude_llm = ChatAnthropicNoTemp(model="claude-haiku-4-5", max_tokens=4096)
ragas_llm_claude = LangchainLLMWrapper(claude_llm)
#ragas_embeddings = LlamaIndexEmbeddingsWrapper(nomic_embed)

metrics = [context_precision, context_recall]
run_config_claude = RunConfig(timeout=180, max_workers=4)

In [9]:
positive_sample

[{'title': 'EmbarkComInc_19991008_S-1A_EX-10.10_6487661_EX-10.10_Co-Branding Agreement',
  'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract',
  'ground_truth': ['CO-BRANDING AGREEMENT'],
  'category': 'Document Name',
  'ground_truth_clean': ['CO-BRANDING AGREEMENT']},
 {'title': 'OASYSMOBILE,INC_07_05_2001-EX-10.17-OUTSOURCING AGREEMENT',
  'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract',
  'ground_truth': ['ASP and Outsourcing Agreement', 'Order Form'],
  'category': 'Document Name',
  'ground_truth_clean': ['ASP and Outsourcing Agreement', 'Order Form']},
 {'title': 'IntegrityFunds_20200121_485BPOS_EX-99.E UNDR CONTR_11948727_EX-99.E UNDR CONTR_Service Agreement',
  'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be rev

## Retrieval evaluation function

`evaluate_retrieval(retrieve_fn, positive_sample, name)` scores any retrieval configuration:
1. it calls `retrieve_fn(question, title)` for each of the 205 examples;
2. it sends the (question, retrieved chunks, reference) triples to ragas in batches of 20;
3. it saves results after every batch in `artifacts/baseline_v3/` and returns the per-example scores plus global and per-category summaries.

**Caution:** results are cached by `name`. Re-running with an existing name resumes from the saved file and skips the judge, so use a new name whenever the configuration changes.


In [11]:
import os
import json
from ragas import EvaluationDataset, evaluate

def evaluate_retrieval(retrieve_fn, positive_sample, name):

    results = []
    BATCH_SIZE = 20
    output_path = f"../artifacts/baseline_v3/retrieval_results_{name}.json"

    for sample in positive_sample:
        question = sample["question"]
        title = sample["title"]
        category = sample["category"]
        reference_context = " ".join(sample["ground_truth_clean"])
        retrieved_context = retrieve_fn(question, title)
        results.append({
            "title": title,
            "category": category,
            "user_input": question,
            "retrieved_contexts": retrieved_context,
            "reference": reference_context,
        })

    if os.path.exists(output_path):
        with open(output_path) as f:
            existing_results = json.load(f)
    else:
        existing_results = []

    start_index = len(existing_results)
    n_batches = (len(results) - 1) // BATCH_SIZE + 1

    for i in range(start_index, len(results), BATCH_SIZE):
        batch = results[i:i + BATCH_SIZE]
        eval_batch = [
            {
                "user_input": r["user_input"],
                "retrieved_contexts": r["retrieved_contexts"],
                "reference": r["reference"],
            }
            for r in batch
        ]
        batch_dataset = EvaluationDataset.from_list(eval_batch)

        print(f"batch {i // BATCH_SIZE + 1}/{n_batches} ({len(batch)} examples)...")

        rs = evaluate(dataset=batch_dataset, metrics=metrics, run_config=run_config_claude, llm=ragas_llm_claude)
        batch_scores = rs.to_pandas().to_dict(orient="records")

        for meta, score in zip(batch, batch_scores):
            existing_results.append({
                "title": meta["title"],
                "category": meta["category"],
                **score,
            })

        with open(output_path, "w") as f:
            json.dump(existing_results, f, indent=2)

    df_scores = pd.DataFrame(existing_results)
    summary_global = df_scores[["context_precision", "context_recall"]].mean()
    summary_by_category = (
        df_scores.groupby("category")[["context_precision", "context_recall"]].mean().sort_values("context_precision"))

    summary_global.to_csv(f"../artifacts/baseline_v3/retrieval_summary_global_{name}.csv")
    summary_by_category.to_csv(f"../artifacts/baseline_v3/retrieval_summary_by_category_{name}.csv")

    return df_scores, summary_global, summary_by_category

## Step 1 — Embedding models (dense search only, `top_k = 3`)

All three models are scored on the same 205 examples, with no tiering and no reranking:

| Model | context_precision | context_recall |
|---|---|---|
| bge-small (baseline v1) | 0.370 | 0.444 |
| bge-large | 0.424 | 0.505 |
| nomic-embed-text | 0.414 | 0.505 |

- bge-small is clearly the weakest.
- bge-large and nomic are essentially tied. In v2, `hit_rate` had ranked nomic clearly first, which does not hold under ragas.
- Both are kept for the hybrid step.
- These scores sit below the v1 production score (0.409 / 0.477) because v1 fetched `Document Name` and `Parties` by position, whereas here every category uses plain semantic search. This is addressed in Step 3.


In [12]:
df_bge_small, summary_global_bge_small, summary_by_category_bge_small = evaluate_retrieval(
    lambda q,t: retrieve_scoped(index_bge_small, q, t),
    positive_sample,
    "bge_small"
)
summary_global_bge_small

batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:40<00:00,  1.02s/it]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:42<00:00,  1.06s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:59<00:00,  1.49s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 10/10 [00:12<00:00,  1.23s/it]


context_precision    0.370325
context_recall       0.444291
dtype: float64

In [13]:
df_bge_large, summary_global_bge_large, summary_by_category_bge_large = evaluate_retrieval(
    lambda q, t: retrieve_scoped(index_bge_large, q, t),
    positive_sample,
    "bge_large"
)
summary_global_bge_large


batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:43<00:00,  1.10s/it]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:43<00:00,  1.10s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [01:03<00:00,  1.59s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 10/10 [00:12<00:00,  1.27s/it]


context_precision    0.423984
context_recall       0.505159
dtype: float64

In [15]:
df_nomic, summary_global_nomic, summary_by_category_nomic = evaluate_retrieval(
    lambda q, t: retrieve_scoped(index_nomic, q, t),
    positive_sample,
    "nomic"
)
summary_global_nomic


context_precision    0.413821
context_recall       0.505495
dtype: float64

## Step 2 — Hybrid search (dense + BM25)

Dense embeddings capture meaning but can miss exact keywords; BM25 (sparse, keyword-based) does the opposite. Hybrid search runs both and lets Qdrant fuse the two result lists.

Two preparation steps:
- **Chunk list (`nodes`)**: rebuilt by scrolling an existing collection (`cuad_bge_small`) with llama-index's `metadata_dict_to_node`. The 14,230 chunks are the same for every embedding model (only the vectors change), so nothing is re-chunked.
- **Hybrid indexes**: a new collection `cuad_bge_large_bm25_hybrid` is built (dense bge-large + sparse `Qdrant/bm25` via fastembed). The nomic hybrid collection already existed from v2 and is simply reloaded.


In [29]:
from llama_index.core.vector_stores.utils import metadata_dict_to_node

def load_nodes_from_collection(client, collection_name):
    nodes = []
    offset = None
    while True:
        records, offset = client.scroll(
            collection_name=collection_name,
            offset=offset,
            limit=1000,
            with_payload=True,
            with_vectors=False
        )
        nodes.extend(metadata_dict_to_node(r.payload) for r in records)
        if offset is None:
            break
    return nodes

nodes = load_nodes_from_collection(qdrant_client, "cuad_bge_small")
len(nodes)

NameError: name 'qdrant_client' is not defined

In [ ]:
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, StorageContext

vector_store_bge_large_hybrid = QdrantVectorStore(
    client=qdrant_client,
    collection_name="cuad_bge_large_bm25_hybrid",
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25"
)

storage_context_bge_large_hybrid = StorageContext.from_defaults(vector_store=vector_store_bge_large_hybrid)
index_bge_large_hybrid = VectorStoreIndex(nodes, storage_context=storage_context_bge_large_hybrid, embed_model=bge_large_embed)

Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 12.14it/s]
/Users/aitbellalhamza/Desktop/RAG-for-contract-understanding-CUAD/.cuad-venv/lib/python3.11/site-packages/llama_index/vector_stores/qdrant/base.py:852: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self._client.create_payload_index(


In [ ]:
vectorstore_nomic_hybrid = QdrantVectorStore(
    collection_name="cuad_nomic_bm25_hybrid",
    client=qdrant_client,
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25"
)
index_nomic_hybrid = VectorStoreIndex.from_vector_store(vectorstore_nomic_hybrid, embed_model=nomic_embed)

## Hybrid retrieval and results

`retrieve_scoped_hybrid` is `retrieve_scoped` with `vector_store_query_mode="hybrid"`: 3 dense candidates + 3 BM25 candidates, fused by Qdrant, still restricted to one contract. Same 205 examples, no reranking yet:

| Configuration | context_precision | context_recall |
|---|---|---|
| bge-large, dense | 0.424 | 0.505 |
| bge-large, hybrid | 0.471 | 0.563 |
| nomic, dense | 0.414 | 0.505 |
| nomic, hybrid | 0.461 | 0.556 |

- Hybrid search helps both models: about +0.05 in precision and +0.05 to +0.06 in recall.
- We continue with **bge-large**. It is marginally ahead, and its 512-token window is not a limitation: it shares the exact tokenizer of bge-small, which was used to size the chunks (longest chunk: 497 tokens), so nothing is truncated.


In [ ]:
def retrieve_scoped_hybrid(index, question, title, top_k=3, sparse_top_k=3):
    filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    retriever = index.as_retriever(
        similarity_top_k=top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=filters,
    )
    nodes = retriever.retrieve(question)
    return [n.node.get_content() for n in nodes]

In [ ]:
df_bge_large_hybrid, summary_global_bge_large_hybrid, summary_by_cat_bge_large_hybrid = evaluate_retrieval(
    lambda q, t: retrieve_scoped_hybrid(index_bge_large_hybrid, q, t),
    positive_sample,
    "bge_large_hybrid",
)
summary_global_bge_large_hybrid


batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:41<00:00,  1.05s/it]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:43<00:00,  1.08s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [01:00<00:00,  1.52s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 10/10 [00:12<00:00,  1.24s/it]


context_precision    0.470732
context_recall       0.563004
dtype: float64

In [ ]:
df_nomic_hybrid, summary_global_nomic_hybrid, summary_by_cat_nomic_hybrid = evaluate_retrieval(
    lambda q, t: retrieve_scoped_hybrid(index_nomic_hybrid, q, t),
    positive_sample,
    "nomic_hybrid",
)
summary_global_nomic_hybrid


batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:42<00:00,  1.06s/it]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:43<00:00,  1.08s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [01:00<00:00,  1.52s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 10/10 [00:12<00:00,  1.27s/it]


context_precision    0.460976
context_recall       0.555866
dtype: float64

## Step 3 — Reranking and positional retrieval (Tier 1)

**Reranking.** Hybrid search now returns 10 dense + 10 BM25 candidates. A cross-encoder reranker (`bge-reranker-large` at this stage) scores each (question, chunk) pair and keeps the best 3.

**`strip_boilerplate`.** Every CUAD question is wrapped in the same generic sentence ("Highlight the parts (if any) of this contract related to ... that should be reviewed by a lawyer"). Before reranking we keep only the quoted category and the "Details" part. The query is shorter (the reranker has a limited window shared by query and chunk) and less noisy.

**Tier 1.** `Document Name` and `Parties` are almost always in the first chunk of a contract, so no search is needed: we return that chunk directly. The category is read from the quoted name inside the question (`extract_category`).

**Result** (bge-large hybrid + reranker + Tier 1): context_precision **0.614**, context_recall **0.641**, versus 0.471 / 0.563 for hybrid alone. This is the largest single jump of the exploration, and both Tier 1 categories reach a precision of 1.0.


In [ ]:
from llama_index.core.postprocessor import SentenceTransformerRerank
import re

reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-large", top_n=3)

def strip_boilerplate(question):
    match = re.search(r'"([^"]+)"', question)
    category_quoted = match.group(0) if match else ""
    details = question.split("Details: ", 1)[1] if "Details: " in question else ""
    return f"{category_quoted}. {details}".strip()


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3166.55it/s]


In [ ]:
TIER1_CATEGORIES = {"Document Name", "Parties"}

def extract_category(question):
    match = re.search(r'"([^"]+)"', question)
    return match.group(1) if match else None

def retrieve_scoped_reranked_tier1(index, question, title, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    category = extract_category(question)

    if category in TIER1_CATEGORIES:
        contract_nodes = [n for n in nodes if n.metadata["title"] == title]
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        return [n.get_content() for n in contract_nodes[:1]]

    filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=filters,
    )
    retrieved_nodes = retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [n.get_content() for n in reranked_nodes]


In [ ]:
df_bge_large_reranked, summary_global_bge_large_reranked, summary_by_cat_bge_large_reranked = evaluate_retrieval(
    lambda q, t: retrieve_scoped_reranked_tier1(index_bge_large_hybrid, q, t),
    positive_sample,
    "bge_large_reranked_tier1",
)
summary_global_bge_large_reranked


batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:31<00:00,  1.25it/s]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:44<00:00,  1.10s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [01:01<00:00,  1.53s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 10/10 [00:11<00:00,  1.20s/it]


context_precision    0.613821
context_recall       0.641188
dtype: float64

## Step 4 — Adaptive `top_k` for weak categories (Tier 3)

Some categories stay weak because their answer is spread over several passages or ranked low. For these, the pool is widened: 15 dense + 15 BM25 candidates, and 5 chunks kept after reranking (instead of 10 → 3).

Categories are chosen from the **per-category** precision of the previous step, not from global averages, and every promotion is checked with a before/after comparison.

**First test (16 categories):** global precision 0.614 → 0.599 and recall 0.641 → 0.677. The gain is mostly recall. Some categories clearly benefit (`Warranty Duration`, `Ip Ownership Assignment`, `Rofr/Rofo/Rofn`, `Cap On Liability`), others do not move or regress (`Minimum Commitment`, `Revenue/Profit Sharing`, `Covenant Not To Sue`). Per-category differences below about 0.1 are within the judge noise, so widening is only kept where the evidence is clear. The final list is fixed later, with the final reranker.


In [ ]:
TIER3_CATEGORIES = {
    "Minimum Commitment", "Volume Restriction", "Most Favored Nation",
    "Exclusivity", "Revenue/Profit Sharing", "Warranty Duration",
    "Covenant Not To Sue", "Ip Ownership Assignment", "Liquidated Damages",
    "Rofr/Rofo/Rofn", "Cap On Liability", "No-Solicit Of Customers", "Affiliate License-Licensor",
    "Price Restrictions", "Non-Disparagement", "Post-Termination Services"
}

def retrieve_scoped_reranked_adaptive(index, question, title, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    category = extract_category(question)

    if category in TIER1_CATEGORIES:
        contract_nodes = [n for n in nodes if n.metadata["title"] == title]
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        return [n.get_content() for n in contract_nodes[:1]]

    if category in TIER3_CATEGORIES:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=filters,
    )
    retrieved_nodes = retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [n.get_content() for n in reranked_nodes]


In [ ]:
df_adaptive, summary_global_adaptive, summary_by_cat_adaptive = evaluate_retrieval(
    lambda q, t: retrieve_scoped_reranked_adaptive(index_bge_large_hybrid, q, t),
    positive_sample,
    "bge_large_adaptive_topk",
)
summary_global_adaptive


batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:30<00:00,  1.31it/s]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:50<00:00,  1.27s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [01:13<00:00,  1.83s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 10/10 [00:19<00:00,  1.96s/it]


context_precision    0.599472
context_recall       0.676875
dtype: float64

## Why do some categories stay near zero? Failure inspection

For the weakest categories (`Minimum Commitment`, `Revenue/Profit Sharing`, `Covenant Not To Sue`), three examples each are inspected. `inspect_category` checks whether the reference text is present (1) among the 15 candidates before reranking and (2) among the 5 chunks kept after reranking.

**Findings (9 examples):**
- reference present among the candidates but dropped by the reranker: 6;
- reference missing from the candidates: 2;
- found and kept: 1.

Reranker scores are close to 0 (at most about 0.01) for every chunk in these categories.

**Two likely causes:**
- CUAD labels can be loose: a "minimum pressure" clause is labelled `Minimum Commitment`, a capital split is labelled `Revenue/Profit Sharing`.
- These categories require conceptual inference, which a general-purpose reranker scores poorly.

Widening the pool would not help, since the right chunk is usually already there. This motivates testing a reranker with a larger context window next.


In [ ]:
import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def is_hit(reference_answers, chunks):
    normalized_chunks = [normalize_text(c) for c in chunks]
    for ref in reference_answers:
        normalized_ref = normalize_text(ref)
        if not normalized_ref:
            continue
        if any(normalized_ref in chunk for chunk in normalized_chunks):
            return True
    return False


In [ ]:
def inspect_category(category, n=3):
    examples = [s for s in positive_sample if s["category"] == category][:n]

    for sample in examples:
        title = sample["title"]
        question = sample["question"]
        reference = sample["ground_truth_clean"]

        filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
        retriever = index_bge_large_hybrid.as_retriever(
            similarity_top_k=15,
            sparse_top_k=15,
            vector_store_query_mode="hybrid",
            filters=filters,
        )
        candidates = retriever.retrieve(question)

        reranker.top_n = 5
        short_question = strip_boilerplate(question)
        reranked = reranker.postprocess_nodes(candidates, query_str=short_question)

        found_before = is_hit(reference, [n.node.get_content() for n in candidates])
        found_after = is_hit(reference, [n.node.get_content() for n in reranked])

        print("=" * 100)
        print(f"CATEGORY: {category}")
        print(f"TITLE: {title}")
        print(f"QUESTION: {question}")
        print(f"REFERENCE: {reference}")
        print(f"Présent dans les 15 candidats (avant rerank) ? {'OUI' if found_before else 'NON'}")
        print(f"Présent dans le top-5 (après rerank) ? {'OUI' if found_after else 'NON'}")
        print("-" * 100)
        for i, node in enumerate(reranked):
            print(f"[reranked #{i+1}] score={node.score:.3f}")
            print(node.get_content()[:400])
            print()


In [ ]:
inspect_category("Minimum Commitment", n=3)


CATEGORY: Minimum Commitment
TITLE: KENTUCKYUTILITIESCO_03_25_2003-EX-10.65-TRANSPORTATION AGREEMENT
QUESTION: Highlight the parts (if any) of this contract related to "Minimum Commitment" that should be reviewed by a lawyer. Details: Is there a minimum order size or minimum amount or units per-time period that one party must buy from the counterparty under the contract?
REFERENCE: ['Transporter shall be obligated to provide such minimum pressures only to the extent that capacity is reserved by Shipper and scheduled by Transporter at the Primary Delivery Point(s) described above.', "In the event Transporter is unable to maintain the minimum pressure(s) described herein but Shipper is still able to take receipt of the scheduled quantity at the Primary Delivery Point(s) described above, then Shipper shall be considered unharmed by Transporter's inability to maintain such minimum pressure(s).", "Transporter shall cause the delivery of natural gas to Shipper at the Shipper's Primary Point 

In [ ]:
inspect_category("Revenue/Profit Sharing", n=3)


CATEGORY: Revenue/Profit Sharing
TITLE: VALENCETECHNOLOGYINC_02_14_2003-EX-10-JOINT VENTURE CONTRACT
QUESTION: Highlight the parts (if any) of this contract related to "Revenue/Profit Sharing" that should be reviewed by a lawyer. Details: Is one party required to share revenue or profit with the counterparty for any technology, goods, or services?
REFERENCE: ["Party A's contribution to the registered capital of the Joint Venture Company shall be Fourteen Million Six Hundred Fifty-One Thousand United States Dollars (US$14,651,000), representing a forty-nine percent (49%) share of the Joint Venture Company's registered capital. Party B's contribution to the registered capital of the Joint Venture Company shall be Fifteen Million Two Hundred Forty Nine Thousand United States Dollars (US$15,249,000), representing a fifty-one percent (51%) share of the Joint Venture Company's registered capital."]
Présent dans les 15 candidats (avant rerank) ? NON
Présent dans le top-5 (après rerank) ? NON


In [ ]:
inspect_category("Covenant Not To Sue", n=3)


CATEGORY: Covenant Not To Sue
TITLE: AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.34_11788308_EX-10.34_Sponsorship Agreement
QUESTION: Highlight the parts (if any) of this contract related to "Covenant Not To Sue" that should be reviewed by a lawyer. Details: Is a party restricted from contesting the validity of the counterparty’s ownership of intellectual property or otherwise bringing a claim against the counterparty for matters unrelated to the contract?
REFERENCE: ["Allied shall not at any time do, or cause to be done, directly or indirectly any act that may impair or tarnish any part of Newegg's goodwill and reputation in the Newegg Marks and the Newegg Products."]
Présent dans les 15 candidats (avant rerank) ? OUI
Présent dans le top-5 (après rerank) ? NON
----------------------------------------------------------------------------------------------------
[reranked #1] score=0.002
2.2 "Including," "Includes" and similar words means "including but not limited to" and shall mea

## Step 5 — A larger reranker: `bge-reranker-v2-m3`

`bge-reranker-large` has a 512-token window shared by the question and the chunk. In a test with a long chunk, the stripped question plus the chunk reached 554 tokens, so the end of the chunk can be cut off. `bge-reranker-v2-m3` accepts 8,192 tokens, so nothing is truncated.

The retrieval functions read the global `reranker`, so reassigning it switches the model everywhere. Same setup as Step 3 (hybrid + reranker + Tier 1, no Tier 3 yet):

| Reranker | context_precision | context_recall |
|---|---|---|
| bge-reranker-large | 0.614 | 0.641 |
| bge-reranker-v2-m3 | 0.625 | 0.647 |

- The global gain is small and hides large per-category swings in both directions: `Exclusivity` and `Cap On Liability` improve a lot, `Ip Ownership Assignment` and `Price Restrictions` drop.
- The conceptual failures seen in the inspection are not fixed, so truncation was not their main cause.
- v2-m3 is kept as the default reranker: slightly better overall and no truncation risk.


In [ ]:
reranker_v2m3 = SentenceTransformerRerank(model="BAAI/bge-reranker-v2-m3", top_n=3)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4979.24it/s]


In [ ]:
reranker = reranker_v2m3

df_bge_large_v2m3, summary_global_v2m3, summary_by_cat_v2m3 = evaluate_retrieval(
    lambda q, t: retrieve_scoped_reranked_tier1(index_bge_large_hybrid, q, t),
    positive_sample,
    "bge_large_reranked_v2m3",
)
summary_global_v2m3


batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:30<00:00,  1.30it/s]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:42<00:00,  1.07s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [01:03<00:00,  1.60s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 10/10 [00:11<00:00,  1.14s/it]


context_precision    0.625203
context_recall       0.646721
dtype: float64

## Step 6 — Tier 3 redone with the new reranker

Widening the pool depends on the reranker, so the selection is repeated with v2-m3. Rule: every category with `context_precision` below 0.6 in the previous run is a candidate (17 categories, `TIER3_CATEGORIES` below), then judged category by category.

Result with all 17 widened: precision 0.625 → 0.617, recall 0.647 → 0.715. Recall improves a lot, precision stays roughly flat.

- **Kept (10, clear gain):** `Ip Ownership Assignment`, `Rofr/Rofo/Rofn`, `Warranty Duration`, `Price Restrictions`, `Affiliate License-Licensor`, `Covenant Not To Sue`, `Termination For Convenience`, `Competitive Restriction Exception`, `Non-Compete`, `Insurance`.
- **Dropped (7, no gain or regression):** `Minimum Commitment`, `Revenue/Profit Sharing`, `Volume Restriction`, `Most Favored Nation`, `Non-Disparagement`, `Liquidated Damages`, `Expiration Date`. The first four stayed near zero in every run.


In [ ]:
TIER3_CATEGORIES = {
    "Volume Restriction", "Minimum Commitment", "Ip Ownership Assignment",
    "Revenue/Profit Sharing", "Rofr/Rofo/Rofn", "Price Restrictions",
    "Most Favored Nation", "Warranty Duration", "Covenant Not To Sue",
    "Liquidated Damages", "Affiliate License-Licensor", "Termination For Convenience",
    "Competitive Restriction Exception", "Non-Compete", "Expiration Date",
    "Non-Disparagement", "Insurance",
}


In [ ]:
df_adaptive_v2m3, summary_global_adaptive_v2m3, summary_by_cat_adaptive_v2m3 = evaluate_retrieval(
    lambda q, t: retrieve_scoped_reranked_adaptive(index_bge_large_hybrid, q, t),
    positive_sample,
    "bge_large_adaptive_v2m3",
)
summary_global_adaptive_v2m3


batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:31<00:00,  1.28it/s]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:48<00:00,  1.20s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [01:13<00:00,  1.83s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 10/10 [00:20<00:00,  2.01s/it]


context_precision    0.616653
context_recall       0.714818
dtype: float64

## Control 1 — Is `strip_boilerplate` still needed with a large-window reranker?

With v2-m3, truncation is no longer a concern, so the stripping might not matter any more. This test passes the raw, unstripped question to the reranker and changes nothing else.

| Reranker input | context_precision | context_recall |
|---|---|---|
| stripped question (baseline) | 0.625 | 0.647 |
| raw question | 0.558 | 0.625 |

Scores drop on both metrics, well beyond the judge noise. Stripping helps because it removes noise, not only because it avoids truncation. It stays.


In [ ]:
def retrieve_scoped_reranked_tier1_raw_question(index, question, title, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    category = extract_category(question)

    if category in TIER1_CATEGORIES:
        contract_nodes = [n for n in nodes if n.metadata["title"] == title]
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        return [n.get_content() for n in contract_nodes[:1]]

    filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=filters,
    )
    retrieved_nodes = retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=question)  # question brute, pas de strip_boilerplate

    return [n.get_content() for n in reranked_nodes]


In [ ]:
reranker = reranker_v2m3  # vérifier qu'on est bien dessus

df_raw_q, summary_global_raw_q, summary_by_cat_raw_q = evaluate_retrieval(
    lambda q, t: retrieve_scoped_reranked_tier1_raw_question(index_bge_large_hybrid, q, t),
    positive_sample,
    "bge_large_v2m3_raw_question",
)
summary_global_raw_q


batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:30<00:00,  1.32it/s]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:42<00:00,  1.07s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:59<00:00,  1.48s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 10/10 [00:11<00:00,  1.16s/it]


context_precision    0.558130
context_recall       0.624925
dtype: float64

## Control 2 — Does another chunking strategy help?

Chunking was only validated with `hit_rate` in v2, where `RecursiveCharacterTextSplitter` chunks (`cuad_recursive_hybrid`, built with nomic) lost against fixed chunking. To change only the chunking, those chunks are read back from the collection and re-embedded with bge-large in a new hybrid collection. They cover the 161 contracts of the evaluation sample.

| Chunking | context_precision | context_recall |
|---|---|---|
| fixed, 512 tokens (baseline) | 0.625 | 0.647 |
| recursive | 0.582 | 0.595 |

Recursive chunking is worse on both metrics, so fixed chunking stays. The sentence-window strategies were not retested: their gap in v2 was large and rebuilding an index for each is costly. The two recursive collections were deleted afterwards to save memory; they can be rebuilt from `baseline_v2.ipynb`.


In [ ]:
recursive_nodes = load_nodes_from_collection(qdrant_client, "cuad_recursive_hybrid")
len(recursive_nodes)


9776

In [ ]:
vector_store_recursive_bge_large = QdrantVectorStore(
    client=qdrant_client,
    collection_name="cuad_recursive_bge_large_hybrid",
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25",
)
storage_context_recursive_bge_large = StorageContext.from_defaults(vector_store=vector_store_recursive_bge_large)
index_recursive_bge_large_hybrid = VectorStoreIndex(recursive_nodes, storage_context=storage_context_recursive_bge_large, embed_model=bge_large_embed)


In [ ]:
def retrieve_scoped_reranked_tier1_recursive(index, question, title, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    category = extract_category(question)

    if category in TIER1_CATEGORIES:
        contract_nodes = [n for n in recursive_nodes if n.metadata["title"] == title]
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        return [n.get_content() for n in contract_nodes[:1]]

    filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=filters,
    )
    retrieved_nodes = retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [n.get_content() for n in reranked_nodes]


In [ ]:
df_recursive, summary_global_recursive, summary_by_cat_recursive = evaluate_retrieval(
    lambda q, t: retrieve_scoped_reranked_tier1_recursive(index_recursive_bge_large_hybrid, q, t),
    positive_sample,
    "recursive_bge_large_v2m3_tier1",
)
summary_global_recursive


batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:31<00:00,  1.27it/s]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:41<00:00,  1.05s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [01:01<00:00,  1.54s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 10/10 [00:12<00:00,  1.21s/it]


context_precision    0.581707
context_recall       0.595110
dtype: float64

## Final retrieval configuration

This cell sets the final Tier 3 list (10 categories) and re-runs the full evaluation with v2-m3, which closes the retrieval exploration.

| Configuration | context_precision | context_recall |
|---|---|---|
| bge-small, dense (v1 baseline) | 0.370 | 0.444 |
| bge-large, dense | 0.424 | 0.505 |
| bge-large, hybrid | 0.471 | 0.563 |
| + bge-reranker-large + Tier 1 | 0.614 | 0.641 |
| bge-reranker-v2-m3 + Tier 1 | 0.625 | 0.647 |
| **+ Tier 3 (10 categories), final** | **0.631** | **0.710** |

**Frozen retrieval pipeline:** fixed 512-token chunking, bge-large, hybrid dense + BM25, `bge-reranker-v2-m3`, `strip_boilerplate`, Tier 1 positional retrieval, Tier 3 widened pool on 10 categories.

For comparison, the v1 production pipeline scored 0.409 / 0.477. The cells that follow use this pipeline unchanged and move on to generation.


In [ ]:
TIER3_CATEGORIES = {
    "Ip Ownership Assignment", "Rofr/Rofo/Rofn", "Warranty Duration",
    "Price Restrictions", "Affiliate License-Licensor", "Covenant Not To Sue",
    "Termination For Convenience", "Competitive Restriction Exception",
    "Non-Compete", "Insurance",
}

reranker = reranker_v2m3  # vérifier qu'on est bien dessus

df_final, summary_global_final, summary_by_cat_final = evaluate_retrieval(
    lambda q, t: retrieve_scoped_reranked_adaptive(index_bge_large_hybrid, q, t),
    positive_sample,
    "bge_large_v2m3_tier3_final",
)
summary_global_final


batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:30<00:00,  1.30it/s]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [00:44<00:00,  1.10s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 40/40 [01:11<00:00,  1.79s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 10/10 [00:12<00:00,  1.26s/it]


context_precision    0.631179
context_recall       0.709787
dtype: float64

## Step 7 — Generation with the frozen retrieval (llama3.2:3b)

The retrieval pipeline is now frozen. To measure what better retrieval alone brings, the first generation run reuses the model and prompt of baseline v1 and v2: `llama3.2:3b` (via Ollama, temperature 0) and the same 6-step prompt (quote the exact clause text, or answer "This clause is not present in this contract." when nothing relevant is found). Only the retrieval changes.

- `generate_answer` retrieves the chunks with the final pipeline and fills the prompt.
- `run_generation` saves after every example and resumes where it stopped, so an interrupted run loses nothing.
- Generation runs on the same fixed samples: 205 positive examples and 118 negative examples.


In [ ]:
negative_sample = checkpoint_v2["negative_sample"]
len(negative_sample)


118

In [ ]:
from llama_index.llms.ollama import Ollama
llm = Ollama(model="llama3.2:3b", request_timeout=180, temperature=0.0)


In [ ]:
from llama_index.core import PromptTemplate

qa_prompt_template = PromptTemplate(
    "Context information from the contract is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a contract review assistant analyzing one contract.\n"
    "Follow these steps:\n"
    "1. Decide whether the context above contains information relevant to the question. "
    "A relevant answer can be a full sentence, a clause, a heading, a title, a date, or a list of names "
    "— not only complete grammatical sentences.\n"
    "2. Do not treat a bare section heading or title (with no accompanying clause text) as an answer just "
    "because it resembles the category name in the question. However, if the context contains actual clause "
    "text — obligations, rights, restrictions, or definitions — that addresses what the question's \"Details\" "
    "describes, that IS valid evidence: quote it normally. Only treat something as \"not present\" when there "
    "is no clause text addressing the details, not merely because a heading happens to be present.\n"
    "3. If the question asks about the Document Name (the name/title of the contract), the answer is "
    "typically a short heading (2-6 words, often in ALL CAPS) located at the very beginning of the context "
    "— either the very first line, or the line immediately following an \"Exhibit X.X\" reference or a "
    "confidentiality/legal notice. Answer with ONLY this short title, do not include the surrounding "
    "paragraphs, recitals, or signature blocks.\n"
    "4. If the question asks about a specific date (such as Agreement Date or Effective Date), answer "
    "ONLY with the explicit date mentioned in a sentence that clearly labels it as such (e.g. \"the Effective Date\", "
    "\"dated as of\"). Do not infer a date from unrelated clauses such as renewal terms, option expirations, "
    "or other unrelated dates.\n"
    "5. Otherwise, if relevant information exists, respond with ONLY the exact text from the context that "
    "answers the question. Quote it directly. Do not paraphrase, do not add explanations or recommendations.\n"
    "6. If nothing in the context is relevant to the question, respond with EXACTLY the following sentence, "
    "word for word, with no rephrasing or additions: \"This clause is not present in this contract.\"\n"
    "Never combine both — give exactly one of the two response types above (step 5 or step 6), never a mix.\n"
    "Question: {query_str}\n"
    "Answer: "
)


In [ ]:
def generate_answer(title, question):
    contexts = retrieve_scoped_reranked_adaptive(index_bge_large_hybrid, question, title)
    context_str = "\n\n".join(contexts)
    prompt = qa_prompt_template.format(context_str=context_str, query_str=question)
    response = llm.complete(prompt)
    return str(response), contexts


In [ ]:
import os
import json
from tqdm import tqdm

def get_tier(category):
    if category in TIER1_CATEGORIES:
        return 1
    if category in TIER3_CATEGORIES:
        return 3
    return 2

def run_generation(sample, is_positive, output_path):
    if os.path.exists(output_path):
        with open(output_path) as f:
            results = json.load(f)
        done_keys = {(r["title"], r["question"]) for r in results}
        remaining = [item for item in sample if (item["title"], item["question"]) not in done_keys]
        print(f"Reprise : {len(results)} deja faits, {len(remaining)} restants")
    else:
        results = []
        remaining = sample

    for item in tqdm(remaining):
        try:
            answer, contexts = generate_answer(item["title"], item["question"])
        except Exception as e:
            answer = f"[ERROR: {type(e).__name__}: {e}]"
            contexts = []

        result = {
            "title": item["title"], "category": item["category"],
            "tier": get_tier(item["category"]),
            "question": item["question"], "answer": answer,
            "contexts": contexts, "is_positive": is_positive,
        }
        result["ground_truth"] = " ".join(item["ground_truth_clean"]) if is_positive else "This clause is not present in this contract."
        results.append(result)
        with open(output_path, "w") as f:
            json.dump(results, f, indent=2)
    return results


In [ ]:
positive_results_v3 = run_generation(positive_sample, is_positive=True, output_path="../artifacts/baseline_v3/positive_results_v3_llama.json")

 36%|███▌      | 73/205 [1:22:54<3:35:43, 98.05s/it] Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13XFamilyCommandBuffer: 0xacd4e4000>
    label = <none> 
    device = <AGXG13XDevice: 0x101d98ba0>
        name = Apple M1 Pro 
    commandQueue = <AGXG13XFamilyCommandQueue: 0x101d9b300>
        label = <none> 
        device = <AGXG13XDevice: 0x101d98ba0>
            name = Apple M1 Pro 
    retainedReferences = 1
 48%|████▊     | 99/205 [2:10:02<3:44:22, 127.00s/it]Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13XFamilyCommandBuffer: 0xacd4e4000>
    label = <none> 
    device = <AGXG13XDevice: 0x101d98ba0>
        name = Apple M1

In [ ]:
negative_results_v3 = run_generation(negative_sample, is_positive=False, output_path="../artifacts/baseline_v3/negative_results_v3_llama.json")

  2%|▏         | 2/118 [02:47<2:42:07, 83.86s/it]


KeyboardInterrupt: 

## Full ragas evaluation of the generated answers

Now that answers exist, all five ragas metrics can be computed (`evaluate_generation`, same batching and resume logic as `evaluate_retrieval`):
- `faithfulness`: is the answer supported by the retrieved chunks?
- `answer_relevancy`: does the answer address the question? This metric needs an embedding model, so nomic is passed to ragas.
- `context_precision` / `context_recall`: same as before, and unchanged in principle since the retrieval is identical.
- `answer_correctness`: how close is the answer to the reference?

| Metric | v1 | v2 | v3 + llama3.2:3b |
|---|---|---|---|
| faithfulness | 0.757 | 0.785 | 0.828 |
| answer_relevancy | 0.614 | 0.574 | 0.624 |
| context_precision | 0.409 | 0.366 | 0.588 |
| context_recall | 0.477 | 0.475 | 0.676 |
| answer_correctness | 0.411 | 0.394 | 0.489 |

All five metrics improve, unlike v2, which only improved `faithfulness`. The two context metrics are a few hundredths below the retrieval-only run (0.631 / 0.710): one example failed during generation and the judge is not deterministic.


In [ ]:
from ragas.metrics import faithfulness, answer_relevancy, answer_correctness
from ragas.embeddings import LlamaIndexEmbeddingsWrapper

ragas_embeddings = LlamaIndexEmbeddingsWrapper(nomic_embed)
metrics_full = [faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness]

def evaluate_generation(results, name):
    clean_results = [r for r in results if not r["answer"].startswith("[ERROR")]
    print(f"{len(clean_results)} usable after removing errors ({len(results) - len(clean_results)} errors)")

    BATCH_SIZE = 20
    output_path = f"../artifacts/baseline_v3/ragas_scores_{name}.json"

    if os.path.exists(output_path):
        with open(output_path) as f:
            existing_scores = json.load(f)
    else:
        existing_scores = []

    start_index = len(existing_scores)
    n_batches = (len(clean_results) - 1) // BATCH_SIZE + 1

    for i in range(start_index, len(clean_results), BATCH_SIZE):
        batch = clean_results[i:i + BATCH_SIZE]
        eval_batch = [
            {
                "user_input": r["question"],
                "response": r["answer"],
                "retrieved_contexts": r["contexts"],
                "reference": r["ground_truth"],
            }
            for r in batch
        ]
        batch_dataset = EvaluationDataset.from_list(eval_batch)

        print(f"batch {i // BATCH_SIZE + 1}/{n_batches} ({len(batch)} examples)...")

        rs = evaluate(dataset=batch_dataset, metrics=metrics_full, llm=ragas_llm_claude, embeddings=ragas_embeddings, run_config=run_config_claude)
        batch_scores = rs.to_pandas().to_dict(orient="records")

        for meta, score in zip(batch, batch_scores):
            existing_scores.append({"title": meta["title"], "category": meta["category"], **score})

        with open(output_path, "w") as f:
            json.dump(existing_scores, f, indent=2)

    df_scores = pd.DataFrame(existing_scores)
    metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness"]
    summary_global = df_scores[metric_cols].mean()
    summary_by_category = df_scores.groupby("category")[metric_cols].mean().sort_values("faithfulness")

    summary_global.to_csv(f"../artifacts/baseline_v3/ragas_summary_global_{name}.csv")
    summary_by_category.to_csv(f"../artifacts/baseline_v3/ragas_summary_by_category_{name}.csv")

    return df_scores, summary_global, summary_by_category


NameError: name 'nomic_embed' is not defined

In [ ]:
df_ragas_v3, summary_global_ragas_v3, summary_by_cat_ragas_v3 = evaluate_generation(positive_results_v3, "v3_llama")
summary_global_ragas_v3

204 usable after removing errors (1 errors)
batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [01:34<00:00,  1.06it/s]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [02:23<00:00,  1.44s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [03:53<00:00,  2.33s/it]


batch 11/11 (4 examples)...


Evaluating: 100%|██████████| 20/20 [00:36<00:00,  1.82s/it]


faithfulness          0.828493
answer_relevancy      0.623819
context_precision     0.587718
context_recall        0.675712
answer_correctness    0.488590
dtype: float64

## Reloading after a kernel restart

After a kernel restart, the next cells reload only what the final pipeline needs: the checkpoint, the Qdrant client, the chunk list (used by the Tier 1 shortcut), the bge-large hybrid index, the reranker, the retrieval function and the LLM. The other exploration indexes are not loaded. The negative generation then resumes automatically where it stopped.


In [ ]:
import os
import json
import pickle
import re
import pandas as pd
from tqdm import tqdm

from qdrant_client import QdrantClient
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
from llama_index.core.vector_stores.utils import metadata_dict_to_node
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core import PromptTemplate
from llama_index.llms.ollama import Ollama


In [ ]:
with open("../artifacts/baseline_v2_checkpoint.pkl", "rb") as f:
    checkpoint_v2 = pickle.load(f)

positive_sample = checkpoint_v2["positive_sample"]
negative_sample = checkpoint_v2["negative_sample"]
len(positive_sample), len(negative_sample)


(205, 118)

In [ ]:
qdrant_client = QdrantClient(path="../artifacts/qdrant_storage")
bge_large_embed = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5")


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3416.91it/s]


In [ ]:
def load_nodes_from_collection(client, collection_name):
    nodes = []
    offset = None
    while True:
        records, offset = client.scroll(
            collection_name=collection_name,
            offset=offset,
            limit=1000,
            with_payload=True,
            with_vectors=False,
        )
        nodes.extend(metadata_dict_to_node(r.payload) for r in records)
        if offset is None:
            break
    return nodes

nodes = load_nodes_from_collection(qdrant_client, "cuad_bge_small")
len(nodes)


14230

In [ ]:
vector_store_bge_large_hybrid = QdrantVectorStore(
    client=qdrant_client,
    collection_name="cuad_bge_large_bm25_hybrid",
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25",
)
index_bge_large_hybrid = VectorStoreIndex.from_vector_store(vector_store_bge_large_hybrid, embed_model=bge_large_embed)


In [ ]:
reranker = SentenceTransformerRerank(model="BAAI/bge-reranker-v2-m3", top_n=3)

TIER1_CATEGORIES = {"Document Name", "Parties"}
TIER3_CATEGORIES = {
    "Ip Ownership Assignment", "Rofr/Rofo/Rofn", "Warranty Duration",
    "Price Restrictions", "Affiliate License-Licensor", "Covenant Not To Sue",
    "Termination For Convenience", "Competitive Restriction Exception",
    "Non-Compete", "Insurance",
}

def extract_category(question):
    match = re.search(r'"([^"]+)"', question)
    return match.group(1) if match else None

def strip_boilerplate(question):
    match = re.search(r'"([^"]+)"', question)
    category_quoted = match.group(0) if match else ""
    details = question.split("Details: ", 1)[1] if "Details: " in question else ""
    return f"{category_quoted}. {details}".strip()

def get_tier(category):
    if category in TIER1_CATEGORIES:
        return 1
    if category in TIER3_CATEGORIES:
        return 3
    return 2


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5893.72it/s]


In [ ]:
def retrieve_scoped_reranked_adaptive(index, question, title, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    category = extract_category(question)

    if category in TIER1_CATEGORIES:
        contract_nodes = [n for n in nodes if n.metadata["title"] == title]
        contract_nodes = sorted(contract_nodes, key=lambda n: n.metadata["chunk_index"])
        return [n.get_content() for n in contract_nodes[:1]]

    if category in TIER3_CATEGORIES:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=filters,
    )
    retrieved_nodes = retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [n.get_content() for n in reranked_nodes]


In [ ]:
llm = Ollama(model="llama3.2:3b", request_timeout=180, temperature=0.0)

qa_prompt_template = PromptTemplate(
    "Context information from the contract is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a contract review assistant analyzing one contract.\n"
    "Follow these steps:\n"
    "1. Decide whether the context above contains information relevant to the question. "
    "A relevant answer can be a full sentence, a clause, a heading, a title, a date, or a list of names "
    "— not only complete grammatical sentences.\n"
    "2. Do not treat a bare section heading or title (with no accompanying clause text) as an answer just "
    "because it resembles the category name in the question. However, if the context contains actual clause "
    "text — obligations, rights, restrictions, or definitions — that addresses what the question's \"Details\" "
    "describes, that IS valid evidence: quote it normally. Only treat something as \"not present\" when there "
    "is no clause text addressing the details, not merely because a heading happens to be present.\n"
    "3. If the question asks about the Document Name (the name/title of the contract), the answer is "
    "typically a short heading (2-6 words, often in ALL CAPS) located at the very beginning of the context "
    "— either the very first line, or the line immediately following an \"Exhibit X.X\" reference or a "
    "confidentiality/legal notice. Answer with ONLY this short title, do not include the surrounding "
    "paragraphs, recitals, or signature blocks.\n"
    "4. If the question asks about a specific date (such as Agreement Date or Effective Date), answer "
    "ONLY with the explicit date mentioned in a sentence that clearly labels it as such (e.g. \"the Effective Date\", "
    "\"dated as of\"). Do not infer a date from unrelated clauses such as renewal terms, option expirations, "
    "or other unrelated dates.\n"
    "5. Otherwise, if relevant information exists, respond with ONLY the exact text from the context that "
    "answers the question. Quote it directly. Do not paraphrase, do not add explanations or recommendations.\n"
    "6. If nothing in the context is relevant to the question, respond with EXACTLY the following sentence, "
    "word for word, with no rephrasing or additions: \"This clause is not present in this contract.\"\n"
    "Never combine both — give exactly one of the two response types above (step 5 or step 6), never a mix.\n"
    "Question: {query_str}\n"
    "Answer: "
)


In [ ]:
def generate_answer(title, question):
    contexts = retrieve_scoped_reranked_adaptive(index_bge_large_hybrid, question, title)
    context_str = "\n\n".join(contexts)
    prompt = qa_prompt_template.format(context_str=context_str, query_str=question)
    response = llm.complete(prompt)
    return str(response), contexts

def run_generation(sample, is_positive, output_path):
    if os.path.exists(output_path):
        with open(output_path) as f:
            results = json.load(f)
        done_keys = {(r["title"], r["question"]) for r in results}
        remaining = [item for item in sample if (item["title"], item["question"]) not in done_keys]
        print(f"Reprise : {len(results)} deja faits, {len(remaining)} restants")
    else:
        results = []
        remaining = sample

    for item in tqdm(remaining):
        try:
            answer, contexts = generate_answer(item["title"], item["question"])
        except Exception as e:
            answer = f"[ERROR: {type(e).__name__}: {e}]"
            contexts = []

        result = {
            "title": item["title"], "category": item["category"],
            "tier": get_tier(item["category"]),
            "question": item["question"], "answer": answer,
            "contexts": contexts, "is_positive": is_positive,
        }
        result["ground_truth"] = " ".join(item["ground_truth_clean"]) if is_positive else "This clause is not present in this contract."
        results.append(result)
        with open(output_path, "w") as f:
            json.dump(results, f, indent=2)
    return results


In [ ]:
negative_results_v3 = run_generation(negative_sample, is_positive=False, output_path="../artifacts/baseline_v3/negative_results_v3_llama.json")

Reprise : 2 deja faits, 116 restants


 81%|████████  | 94/116 [54:15<14:23, 39.27s/it]  Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13XFamilyCommandBuffer: 0x867138000>
    label = <none> 
    device = <AGXG13XDevice: 0x102fce5f0>
        name = Apple M1 Pro 
    commandQueue = <AGXG13XFamilyCommandQueue: 0x102fd0e40>
        label = <none> 
        device = <AGXG13XDevice: 0x102fce5f0>
            name = Apple M1 Pro 
    retainedReferences = 1
 97%|█████████▋| 112/116 [1:07:21<03:33, 53.48s/it]Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG13XFamilyCommandBuffer: 0x867138000>
    label = <none> 
    device = <AGXG13XDevice: 0x102fce5f0>
        name = Apple M1 Pro 

## False positives on negative examples

For the 118 negative examples the clause does not exist, so the right answer is always "not present". A keyword check on the answer (`not present`, `not specified`, `not found`, ...) separates true negatives from false positives (the model claims a clause exists). No LLM judge is involved.

| Pipeline | False positive rate |
|---|---|
| v1 | 64.4% (76/118) |
| v2 | 66.1% (78/118) |
| v3 + llama3.2:3b | **71.2%** (84/118) |

Better retrieval did not reduce hallucinations, it made them slightly worse. A likely explanation: the improved retrieval surfaces chunks that are closer to the category but still wrong, and a 3B model takes them for evidence. The problem seems to be on the generation side.


In [ ]:
from collections import Counter

ABSENCE_PHRASES = [
    "not present", "not specified", "not mentioned", "does not contain",
    "no clause", "not applicable", "not included", "not found", "n/a", "absent"
]

def classify_negative(answer):
    if answer.startswith("[ERROR"):
        return "error"
    answer_lower = answer.lower()
    if any(phrase in answer_lower for phrase in ABSENCE_PHRASES):
        return "true negative"
    return "false positive"

for item in negative_results_v3:
    item["classification"] = classify_negative(item["answer"])

counts_v3 = Counter(item["classification"] for item in negative_results_v3)
print(counts_v3)

total_classifiable = counts_v3["true negative"] + counts_v3["false positive"]
fp_rate = counts_v3["false positive"] / total_classifiable * 100
print(f"False positive rate v3: {fp_rate:.1f}% ({counts_v3['false positive']}/{total_classifiable})")


Counter({'false positive': 84, 'true negative': 34})
False positive rate v3: 71.2% (84/118)


## Idea tested: a confidence threshold on reranker scores

If chunks with a low reranker score were dropped, negative examples might end up with little or no context, and the model would answer "not present". Reranker scores are extracted by position (1st, 2nd, ... chunk) for positive and negative examples, then two thresholds are computed per position.

| Threshold | Positives kept | Negatives kept |
|---|---|---|
| optimal (Youden's J) | about 51–52% at positions 0–2 | 14–28% |
| soft (85% of positives kept) | 85% | 59–70% |

- The optimal threshold would discard about half of the correct chunks, which destroys the recall gained so far.
- The soft threshold still keeps 59–70% of the negative chunks at each position. For an example to end up with an empty context, all its positions must be filtered: under 5% of the negative examples in the best case.

Not pursued: the threshold does not separate positives from negatives well enough (same conclusion as in v2). The next step attacks the generation side instead.


In [ ]:
def get_reranked_with_scores(index, question, title, retrieve_top_k=10, sparse_top_k=10, rerank_top_n=3):
    category = extract_category(question)

    if category in TIER1_CATEGORIES:
        return None  # pas de reranker sur Tier 1, rien à mesurer

    if category in TIER3_CATEGORIES:
        retrieve_top_k = 15
        sparse_top_k = 15
        rerank_top_n = 5

    filters = MetadataFilters(filters=[ExactMatchFilter(key="title", value=title)])
    retriever = index.as_retriever(
        similarity_top_k=retrieve_top_k,
        sparse_top_k=sparse_top_k,
        vector_store_query_mode="hybrid",
        filters=filters,
    )
    retrieved_nodes = retriever.retrieve(question)

    reranker.top_n = rerank_top_n
    short_question = strip_boilerplate(question)
    reranked_nodes = reranker.postprocess_nodes(retrieved_nodes, query_str=short_question)

    return [(n.get_content(), n.score) for n in reranked_nodes]


In [ ]:
from collections import defaultdict

positive_scores_by_position = defaultdict(list)
for sample in positive_sample:
    result = get_reranked_with_scores(index_bge_large_hybrid, sample["question"], sample["title"])
    if result is None:
        continue
    for i, (text, score) in enumerate(result):
        positive_scores_by_position[i].append(score)

negative_scores_by_position = defaultdict(list)
for sample in negative_sample:
    result = get_reranked_with_scores(index_bge_large_hybrid, sample["question"], sample["title"])
    if result is None:
        continue
    for i, (text, score) in enumerate(result):
        negative_scores_by_position[i].append(score)


In [ ]:
import statistics

for i in sorted(positive_scores_by_position):
    pos = positive_scores_by_position[i]
    neg = negative_scores_by_position.get(i, [])
    print(f"Position {i}: positif médiane={statistics.median(pos):.3f} (n={len(pos)}) | "
          f"négatif médiane={statistics.median(neg):.3f} (n={len(neg)})" if neg else
          f"Position {i}: positif médiane={statistics.median(pos):.3f} (n={len(pos)}) | pas de négatif à cette position")


Position 0: positif médiane=0.091 (n=195) | négatif médiane=0.016 (n=117)
Position 1: positif médiane=0.022 (n=194) | négatif médiane=0.010 (n=114)
Position 2: positif médiane=0.014 (n=194) | négatif médiane=0.008 (n=111)
Position 3: positif médiane=0.010 (n=50) | négatif médiane=0.005 (n=28)
Position 4: positif médiane=0.008 (n=50) | négatif médiane=0.003 (n=28)


In [ ]:
def youden_threshold(pos_scores, neg_scores):
    all_scores = sorted(set(pos_scores + neg_scores))
    best_threshold = None
    best_j = -1
    for t in all_scores:
        tpr = sum(s >= t for s in pos_scores) / len(pos_scores)
        fpr = sum(s >= t for s in neg_scores) / len(neg_scores)
        j = tpr - fpr
        if j > best_j:
            best_j = j
            best_threshold = t
    return best_threshold, best_j

for i in sorted(positive_scores_by_position):
    pos = positive_scores_by_position[i]
    neg = negative_scores_by_position.get(i, [])
    if not neg:
        continue
    t, j = youden_threshold(pos, neg)
    tpr = sum(s >= t for s in pos) / len(pos)
    fpr = sum(s >= t for s in neg) / len(neg)
    print(f"Position {i}: seuil={t:.4f} (J={j:.3f}, garde {tpr:.0%} des positifs, garde {fpr:.0%} des négatifs)")


Position 0: seuil=0.0904 (J=0.371, garde 51% des positifs, garde 14% des négatifs)
Position 1: seuil=0.0210 (J=0.252, garde 52% des positifs, garde 26% des négatifs)
Position 2: seuil=0.0138 (J=0.231, garde 51% des positifs, garde 28% des négatifs)
Position 3: seuil=0.0027 (J=0.269, garde 84% des positifs, garde 57% des négatifs)
Position 4: seuil=0.0047 (J=0.291, garde 72% des positifs, garde 43% des négatifs)


In [ ]:
import numpy as np

def soft_threshold(pos_scores, keep_fraction=0.85):
    return np.percentile(pos_scores, (1 - keep_fraction) * 100)

for i in sorted(positive_scores_by_position):
    pos = positive_scores_by_position[i]
    neg = negative_scores_by_position.get(i, [])
    if not neg:
        continue
    t = soft_threshold(pos)
    tpr = sum(s >= t for s in pos) / len(pos)
    fpr = sum(s >= t for s in neg) / len(neg)
    print(f"Position {i}: seuil doux={t:.4f} (garde {tpr:.0%} des positifs, garde {fpr:.0%} des négatifs)")


Position 0: seuil doux=0.0119 (garde 85% des positifs, garde 59% des négatifs)
Position 1: seuil doux=0.0069 (garde 85% des positifs, garde 61% des négatifs)
Position 2: seuil doux=0.0042 (garde 85% des positifs, garde 70% des négatifs)
Position 3: seuil doux=0.0025 (garde 84% des positifs, garde 68% des négatifs)
Position 4: seuil doux=0.0020 (garde 84% des positifs, garde 68% des négatifs)


## Step 8 — A larger generation model: qwen2.5:7b

The false positive rate above points to the generation model, so the next test swaps only the LLM: `qwen2.5:7b` (about 5.5 GB in memory), with the same prompt and the same frozen retrieval.

Memory is the constraint on a 16 GB laptop, so the run is prepared carefully:
- unused Qdrant collections were deleted (the local Qdrant client loads every collection of the storage folder), which brought the Python kernel from about 10 GB down to about 6.5 GB;
- a first test on 10 negative examples was run while watching memory, before launching the full runs;
- generation is resumable, so it can be stopped and restarted at any time.

The prompt below is identical to the one used with llama3.2:3b.


In [ ]:
qa_prompt_template = PromptTemplate(
    "Context information from the contract is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a contract review assistant analyzing one contract.\n"
    "Follow these steps:\n"
    "1. Decide whether the context above contains information relevant to the question. "
    "A relevant answer can be a full sentence, a clause, a heading, a title, a date, or a list of names "
    "— not only complete grammatical sentences.\n"
    "2. Do not treat a bare section heading or title (with no accompanying clause text) as an answer just "
    "because it resembles the category name in the question. However, if the context contains actual clause "
    "text — obligations, rights, restrictions, or definitions — that addresses what the question's \"Details\" "
    "describes, that IS valid evidence: quote it normally. Only treat something as \"not present\" when there "
    "is no clause text addressing the details, not merely because a heading happens to be present.\n"
    "3. If the question asks about the Document Name (the name/title of the contract), the answer is "
    "typically a short heading (2-6 words, often in ALL CAPS) located at the very beginning of the context "
    "— either the very first line, or the line immediately following an \"Exhibit X.X\" reference or a "
    "confidentiality/legal notice. Answer with ONLY this short title, do not include the surrounding "
    "paragraphs, recitals, or signature blocks.\n"
    "4. If the question asks about a specific date (such as Agreement Date or Effective Date), answer "
    "ONLY with the explicit date mentioned in a sentence that clearly labels it as such (e.g. \"the Effective Date\", "
    "\"dated as of\"). Do not infer a date from unrelated clauses such as renewal terms, option expirations, "
    "or other unrelated dates.\n"
    "5. Otherwise, if relevant information exists, respond with ONLY the exact text from the context that "
    "answers the question. Quote it directly. Do not paraphrase, do not add explanations or recommendations.\n"
    "6. If nothing in the context is relevant to the question, respond with EXACTLY the following sentence, "
    "word for word, with no rephrasing or additions: \"This clause is not present in this contract.\"\n"
    "Never combine both — give exactly one of the two response types above (step 5 or step 6), never a mix.\n"
    "Question: {query_str}\n"
    "Answer: "
)

In [ ]:
llm = Ollama(model="qwen2.5:7b", request_timeout=180, temperature=0.0)

In [ ]:
small_test_sample = negative_sample[:10]
test_results_qwen = run_generation(small_test_sample, is_positive=False, output_path="../artifacts/baseline_v3/test_qwen_small.json")


100%|██████████| 10/10 [02:22<00:00, 14.24s/it]


In [ ]:
negative_results_v3_qwen = run_generation(
    negative_sample[:118],
    is_positive=False,
    output_path="../artifacts/baseline_v3/negative_results_v3_qwen.json"
)


Reprise : 59 deja faits, 59 restants


100%|██████████| 59/59 [12:46<00:00, 13.00s/it]


In [ ]:
positive_results_v3_qwen = run_generation(
    positive_sample,
    is_positive=True,
    output_path="../artifacts/baseline_v3/positive_results_v3_qwen.json"
)

Reprise : 100 deja faits, 105 restants


100%|██████████| 105/105 [28:11<00:00, 16.11s/it]


## Results with qwen2.5:7b: far fewer hallucinations, but many abstentions

**Negative examples:** the false positive rate falls from 71.2% to **3.4%** (4/118) with the same retrieval. The generation model was the problem.

**Positive examples** (same 205 examples, ragas):

| Metric | llama3.2:3b | qwen2.5:7b |
|---|---|---|
| faithfulness | 0.828 | 0.573 |
| answer_relevancy | 0.624 | 0.654 |
| context_precision | 0.588 | 0.625 |
| context_recall | 0.676 | 0.708 |
| answer_correctness | 0.489 | 0.403 |
| "not present" answers | 13% | 49% |

- The two context metrics should be identical (same retrieval): the gap of about 0.035 is the judge noise.
- qwen answers "not present" on 49% of the positive examples, where the clause does exist. When it does answer, it is better than llama (`answer_correctness` 0.658 vs 0.530).
- Part of the `faithfulness` drop is an artifact: ragas scores "not present" answers very low (about 0.23), so more abstentions lower the average mechanically.


In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LlamaIndexEmbeddingsWrapper
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness
from ragas.run_config import RunConfig
from ragas import EvaluationDataset, evaluate
from llama_index.embeddings.ollama import OllamaEmbedding

class ChatAnthropicNoTemp(ChatAnthropic):
    def __setattr__(self, name, value):
        if name == "temperature":
            return
        super().__setattr__(name, value)

claude_llm = ChatAnthropicNoTemp(model="claude-haiku-4-5", max_tokens=4096)
ragas_llm_claude = LangchainLLMWrapper(claude_llm)

nomic_embed = OllamaEmbedding(model_name="nomic-embed-text")
ragas_embeddings = LlamaIndexEmbeddingsWrapper(nomic_embed)

metrics_full = [faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness]
run_config_claude = RunConfig(timeout=180, max_workers=4)


In [ ]:
def evaluate_generation(results, name):
    clean_results = [r for r in results if not r["answer"].startswith("[ERROR")]
    print(f"{len(clean_results)} usable after removing errors ({len(results) - len(clean_results)} errors)")

    BATCH_SIZE = 20
    output_path = f"../artifacts/baseline_v3/ragas_scores_{name}.json"

    if os.path.exists(output_path):
        with open(output_path) as f:
            existing_scores = json.load(f)
    else:
        existing_scores = []

    start_index = len(existing_scores)
    n_batches = (len(clean_results) - 1) // BATCH_SIZE + 1

    for i in range(start_index, len(clean_results), BATCH_SIZE):
        batch = clean_results[i:i + BATCH_SIZE]
        eval_batch = [
            {
                "user_input": r["question"],
                "response": r["answer"],
                "retrieved_contexts": r["contexts"],
                "reference": r["ground_truth"],
            }
            for r in batch
        ]
        batch_dataset = EvaluationDataset.from_list(eval_batch)

        print(f"batch {i // BATCH_SIZE + 1}/{n_batches} ({len(batch)} examples)...")

        rs = evaluate(dataset=batch_dataset, metrics=metrics_full, llm=ragas_llm_claude, embeddings=ragas_embeddings, run_config=run_config_claude)
        batch_scores = rs.to_pandas().to_dict(orient="records")

        for meta, score in zip(batch, batch_scores):
            existing_scores.append({"title": meta["title"], "category": meta["category"], **score})

        with open(output_path, "w") as f:
            json.dump(existing_scores, f, indent=2)

    df_scores = pd.DataFrame(existing_scores)
    metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall", "answer_correctness"]
    summary_global = df_scores[metric_cols].mean()
    summary_by_category = df_scores.groupby("category")[metric_cols].mean().sort_values("faithfulness")

    summary_global.to_csv(f"../artifacts/baseline_v3/ragas_summary_global_{name}.csv")
    summary_by_category.to_csv(f"../artifacts/baseline_v3/ragas_summary_by_category_{name}.csv")

    return df_scores, summary_global, summary_by_category


In [ ]:
df_ragas_v3_qwen, summary_global_ragas_v3_qwen, summary_by_cat_ragas_v3_qwen = evaluate_generation(
    positive_results_v3_qwen, "v3_qwen"
)
summary_global_ragas_v3_qwen

205 usable after removing errors (0 errors)
batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [01:15<00:00,  1.32it/s]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [01:45<00:00,  1.06s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [02:34<00:00,  1.54s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 25/25 [00:36<00:00,  1.44s/it]


faithfulness          0.573143
answer_relevancy      0.654416
context_precision     0.625224
context_recall        0.708027
answer_correctness    0.403325
dtype: float64

## Are qwen's abstentions justified?

An abstention is "justified" when the retrieved chunks really do not contain the answer, and "over-cautious" when they do. The `context_recall` of the same example is used as the test:

| qwen "not present" answers (100) | Count |
|---|---|
| `context_recall` < 0.3: the context lacks the answer, abstaining is right | 41 |
| between 0.3 and 0.7: partial context | 8 |
| `context_recall` ≥ 0.7: the context has the answer, abstaining is a miss | **51** |

The justified abstentions match the categories where retrieval fails (`Minimum Commitment`, `Volume Restriction`).

The 51 misses are spread over 27 of the 41 categories, so this is not a category-specific weakness. The three date categories account for only 6 of them, which rules out the date rule of the prompt as the main cause. The cells below show examples where the reference text is literally inside a chunk given to qwen.


In [ ]:
import json
import re
import pandas as pd

with open("../artifacts/baseline_v3/ragas_scores_v3_qwen.json") as f:
    df_qwen = pd.DataFrame(json.load(f))

df_qwen["is_not_present"] = df_qwen["response"].str.lower().str.contains("not present")
candidates = df_qwen[df_qwen["is_not_present"] & (df_qwen["context_recall"] >= 0.7)]
len(candidates)


51

In [ ]:
def normalize_text(text):
    return re.sub(r"\s+", " ", text.lower()).strip()

def show_example(row):
    question = row["user_input"]
    details = question.split("Details: ", 1)[1] if "Details: " in question else question
    reference = row["reference"]
    reference_start = normalize_text(reference)[:80]

    print("=" * 100)
    print("CATEGORY:", row["category"])
    print("DETAILS:", details)
    print("REFERENCE:", reference[:300])

    found = False
    for i, chunk in enumerate(row["retrieved_contexts"]):
        chunk_norm = normalize_text(chunk)
        pos = chunk_norm.find(reference_start)
        if pos != -1:
            print(f"-> référence trouvée dans le chunk #{i} :")
            print("   ...", chunk_norm[max(0, pos - 150): pos + 400], "...")
            found = True
            break
    if not found:
        print("-> début de la référence non trouvé mot pour mot dans les chunks")

    print("QWEN:", row["response"][:150])


In [ ]:
picked = candidates.drop_duplicates(subset="category").head(6)
for _, row in picked.iterrows():
    show_example(row)


CATEGORY: Parties
DETAILS: The two or more parties who signed the contract
REFERENCE: Accuray Siemens SIEMENS AKTIENGESELLSCHAFT ACCURAY INCORPORATED
-> début de la référence non trouvé mot pour mot dans les chunks
QWEN: This clause is not present in this contract.
CATEGORY: Agreement Date
DETAILS: The date of the contract
REFERENCE: July 16, 2012
-> référence trouvée dans le chunk #1 :
   ... nstitute one and the same agreement. remainder of page intentionally left blank. signature page(s) to follow. tender offer statement on schedule to - july 16, 2012 exhibit (d)(i) sponsorship agreement - july 16, 2012 remainder of page intentionally left blank. tender offer statement on schedule to - july 16, 2012 exhibit (d)(i) sponsorship agreement - july 16, 2012 in witness whereof, this agreement is executed and effective as of the effective date. purchaser: peek investments llc date: july 16, 2012 by: /s/ michael onghai name: michael onghai ...
QWEN: This clause is not present in this contrac

In [ ]:
df_qwen["over_cautious"] = df_qwen["is_not_present"] & (df_qwen["context_recall"] >= 0.7)

by_cat = df_qwen.groupby("category").agg(
    n=("over_cautious", "size"),
    abstentions=("is_not_present", "sum"),
    over_cautious=("over_cautious", "sum"),
)
by_cat.sort_values(["over_cautious", "abstentions"], ascending=False)


,n,abstentions,over_cautious
category,,,
Expiration Date,5,5,4
Termination For Convenience,5,4,4
Rofr/Rofo/Rofn,5,5,3
Uncapped Liability,5,4,3
Change Of Control,5,3,3
Non-Compete,5,3,3
Price Restrictions,5,3,3
Renewal Term,5,3,3
Agreement Date,5,3,2


In [ ]:
DATE_CATEGORIES = {"Agreement Date", "Effective Date", "Expiration Date"}
df_qwen["family"] = df_qwen["category"].apply(lambda c: "dates" if c in DATE_CATEGORIES else "autres")

family_stats = df_qwen.groupby("family").agg(
    n=("over_cautious", "size"),
    abstentions=("is_not_present", "sum"),
    over_cautious=("over_cautious", "sum"),
)
family_stats["taux_over_cautious"] = (family_stats["over_cautious"] / family_stats["n"]).round(2)
family_stats


,n,abstentions,over_cautious,taux_over_cautious
family,,,,
autres,190,92,45,0.24
dates,15,8,6,0.40


In [ ]:
for cat in ["Termination For Convenience", "Uncapped Liability", "Change Of Control"]:
    subset = df_qwen[(df_qwen["category"] == cat) & df_qwen["over_cautious"]].head(2)
    for _, row in subset.iterrows():
        show_example(row)


CATEGORY: Termination For Convenience
DETAILS: Can a party terminate this  contract without cause (solely by giving a notice and allowing a waiting  period to expire)?
REFERENCE: Notwithstanding Article 9.3., [CERES/IGER] will have the right to terminate this Agreement including the Program at any time with [three (3)] months' prior written notice.
-> référence trouvée dans le chunk #4 :
   ...  last report provided for in this agreement, or (iii) the dates on which the last payment due by [ceres/iger] pursuant to this agreement is made. 9.2 notwithstanding article 9.3., [ceres/iger] will have the right to terminate this agreement including the program at any time with [three (3)] months' prior written notice. in case of such early termination, [ceres/iger] will pay the remuneration provided in this agreement up to the effective date of termination, and all costs relating to the implementation of this agreement which [party] irrevocab ...
QWEN: This clause is not present in this contra

## Why does qwen refuse? Asking it to explain

The examples above show clauses that answer the question almost word for word, yet qwen abstains. To understand the decision, `diagnose` asks qwen to write a short "Reasoning:" before its answer, on 6 of these cases.

- With a reasoning step, qwen answers in 4 of the 6 cases. The refusal looks like a quick, shallow decision rather than a lack of ability.
- 5 of the 6 explanations say the clause is not *explicitly* mentioned. The prompt asks for explicitness only for dates, so qwen applies a stricter standard on its own, and seems to look for the category name rather than the meaning of the clause.
- 2 of the 6 explanations confuse the numbered steps of the prompt with the section numbers of the contract.

These are after-the-fact explanations on 6 cases: they are hints, not proof.


In [ ]:
def diagnose(row):
    context_str = "\n\n".join(row["retrieved_contexts"])
    prompt = qa_prompt_template.format(context_str=context_str, query_str=row["user_input"])
    idx = prompt.rfind("Answer:")
    prompt = prompt[:idx] + (
        "Before answering, write one or two sentences starting with 'Reasoning:' explaining "
        "which passage (if any) you considered, and which of the numbered steps made you "
        "decide to quote it or to answer 'not present'. Then write 'Answer:' followed by your answer.\n"
        "Reasoning:"
    )
    return str(llm.complete(prompt))

rows = []
for cat in ["Termination For Convenience", "Uncapped Liability", "Change Of Control"]:
    subset = df_qwen[(df_qwen["category"] == cat) & df_qwen["over_cautious"]].head(2)
    rows += [r for _, r in subset.iterrows()]

for row in rows:
    print("=" * 80)
    print(row["category"])
    print(diagnose(row))


Termination For Convenience
Reasoning: The relevant clause in this contract is found under "16. TERM AND TERMINATION," which discusses various termination scenarios but does not explicitly mention a termination for convenience without cause. Steps 2 and 15.3 indicate that the agreement should be interpreted according to New York law, which may provide additional context on termination clauses.

Answer: This clause is not present in this contract.
Termination For Convenience
Reasoning: The contract does not explicitly mention "Termination For Convenience" but provides termination rights for both parties under specific conditions. Steps 2 and 4 guided the decision that no such clause is present, as the relevant sections (Steps 1 and 3) do not address a party's ability to terminate without cause.

Answer: This clause is not present in this contract.
Uncapped Liability
Reasoning: The context provided does not contain any explicit provisions related to uncapped liability for breaches such a

## Test 1 — Rewriting the prompt (v2)

Prompt v2 uses rules A to F instead of numbered steps, softens the date rule, asks to judge by meaning rather than by wording, and accepts clauses written as exceptions. It is tested without re-running retrieval (the retrieved chunks are already stored), on the 51 over-cautious positives and the 118 negatives.

| Prompt | Over-cautious answers recovered (of 51) | False positives |
|---|---|---|
| v1 | 0 | 3.4% (4/118) |
| v2 | 9 | 5.1% (6/118) |

The gain is small and mostly limited to date and renewal categories: `Termination For Convenience`, `Change Of Control` or `Non-Compete` do not move. Rewording the rules is not enough. Combining prompt v2 with the reasoning step is reported in Test 2.


In [ ]:
from llama_index.core import PromptTemplate
from tqdm import tqdm

qa_prompt_template_v2 = PromptTemplate(
    "Context information from the contract is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "You are a contract review assistant analyzing one contract.\n"
    "Apply rules A to F below. They are instructions for you, not sections of the contract.\n"
    "Rule A. Decide whether ANY passage of the context addresses what the question's \"Details\" describe. "
    "Judge by meaning, not by wording: a clause can answer the Details without using the category name or the "
    "words of the question, and it does not need to state its conclusion explicitly. A relevant passage can be a "
    "full clause, a sentence, a date, or a list of names. A clause written as an exception or carve-out "
    "(for example \"shall not limit...\" or \"except in cases of...\") counts when it defines the situation the "
    "Details ask about.\n"
    "Rule B. Do not treat a bare section heading or title (with no accompanying clause text) as an answer just "
    "because it resembles the category name. However, if the context contains actual clause text — obligations, "
    "rights, restrictions, or definitions — that addresses the Details, that IS valid evidence: quote it. "
    "Do not answer \"not present\" merely because the category name or the exact words of the question do not appear.\n"
    "Rule C. If the question asks about the Document Name (the name/title of the contract), the answer is "
    "typically a short heading (2-6 words, often in ALL CAPS) located at the very beginning of the context "
    "— either the very first line, or the line immediately following an \"Exhibit X.X\" reference or a "
    "confidentiality/legal notice. Answer with ONLY this short title, do not include the surrounding "
    "paragraphs, recitals, or signature blocks.\n"
    "Rule D. If the question asks about a specific date (such as Agreement Date, Effective Date or Expiration Date), "
    "answer with the sentence that states it: a date labeled as such (e.g. \"the Effective Date\", \"dated as of\"), "
    "the date in the agreement header or signature block for the Agreement Date, or the sentence giving the length "
    "of the term for the Expiration Date. Do not take a date from unrelated clauses such as option expirations or "
    "payment deadlines.\n"
    "Rule E. If relevant information exists, respond with ONLY the exact text from the context that answers the "
    "question. Quote it directly. Do not paraphrase, do not add explanations or recommendations.\n"
    "Rule F. Only if no passage of the context addresses the Details, respond with EXACTLY the following sentence, "
    "word for word, with no rephrasing or additions: \"This clause is not present in this contract.\"\n"
    "Give exactly one of the two response types (Rule E or Rule F), never a mix.\n"
    "Question: {query_str}\n"
    "Answer: "
)

def regenerate(contexts, question, template):
    prompt = template.format(context_str="\n\n".join(contexts), query_str=question)
    return str(llm.complete(prompt))


In [ ]:
over_cautious_df = df_qwen[df_qwen["over_cautious"]].copy()

new_answers = []
for _, row in tqdm(over_cautious_df.iterrows(), total=len(over_cautious_df)):
    new_answers.append(regenerate(row["retrieved_contexts"], row["user_input"], qa_prompt_template_v2))
over_cautious_df["new_answer"] = new_answers

still_refused = over_cautious_df["new_answer"].str.lower().str.contains("not present")
print(f"Récupérées : {(~still_refused).sum()}/{len(over_cautious_df)}")
print(over_cautious_df.loc[~still_refused, "category"].value_counts().to_string())


100%|██████████| 51/51 [10:09<00:00, 11.94s/it]

Récupérées : 9/51
category
Expiration Date       3
Renewal Term          3
Parties               1
Agreement Date        1
Uncapped Liability    1


In [ ]:
import json
from collections import Counter

with open("../artifacts/baseline_v3/negative_results_v3_qwen.json") as f:
    negative_results_v3_qwen = json.load(f)

ABSENCE_PHRASES = [
    "not present", "not specified", "not mentioned", "does not contain",
    "no clause", "not applicable", "not included", "not found", "n/a", "absent"
]

def classify_negative(answer):
    answer_lower = answer.lower()
    return "true negative" if any(p in answer_lower for p in ABSENCE_PHRASES) else "false positive"

new_negative = []
for item in tqdm(negative_results_v3_qwen):
    new_answer = regenerate(item["contexts"], item["question"], qa_prompt_template_v2)
    new_negative.append({**item, "answer": new_answer, "classification": classify_negative(new_answer)})

counts = Counter(x["classification"] for x in new_negative)
print(counts)
print(f"Taux de FP prompt v2 : {counts['false positive'] / len(new_negative):.1%} (prompt v1 : 3.4%)")


100%|██████████| 118/118 [20:20<00:00, 10.35s/it]

Counter({'true negative': 112, 'false positive': 6})
Taux de FP prompt v2 : 5.1% (prompt v1 : 3.4%)


## Test 2 — Reasoning before answering

The prompt stays v1, but qwen is asked to write one or two sentences starting with "Reasoning:" before "Answer:". `regenerate_with_reasoning` keeps only the text after `Answer:` as the final answer. False positives are classified on the final answer only, because the reasoning itself often contains phrases like "does not contain" that would fool the keyword check.

Same test set as before (retrieved chunks already stored):

| Variant | Over-cautious answers recovered (of 51) | False positives |
|---|---|---|
| prompt v1 | 0 | 3.4% (4/118) |
| prompt v2 | 9 | 5.1% (6/118) |
| **prompt v1 + reasoning** | **24** | **9.3%** (11/118) |
| prompt v2 + reasoning | 22 | 21% (about 25/118) |

Combining prompt v2 with the reasoning step was tested on both samples. On the negatives, false positives rise to 21% because the two loosenings compound. On the positives, `answer_correctness` is 0.464 against 0.479 for v1 + reasoning — within the judge noise, no real gain — and the abstention rate does not move either (74 vs 72 "not present" answers). Rejected: the extra false positives buy nothing.


Of the 24 recovered answers, 17 quote the reference text (the check is approximate: a valid clause other than the reference is counted as a miss).

Trade-off: 17 more clauses correctly quoted against 7 more false positives, with the false positive rate still under 10%. This variant is worth a full run.


In [ ]:
def regenerate_with_reasoning(contexts, question, template):
    prompt = template.format(context_str="\n\n".join(contexts), query_str=question)
    idx = prompt.rfind("Answer:")
    prompt = prompt[:idx] + (
        "First write one or two sentences starting with 'Reasoning:' saying which passage of the context, "
        "if any, addresses the Details. Then write 'Answer:' followed by your answer, following the rules above.\n"
        "Reasoning:"
    )
    raw = str(llm.complete(prompt))
    final = raw.split("Answer:", 1)[1].strip() if "Answer:" in raw else raw.strip()
    return final, raw


In [ ]:
finals, raws = [], []
for _, row in tqdm(over_cautious_df.iterrows(), total=len(over_cautious_df)):
    final, raw = regenerate_with_reasoning(row["retrieved_contexts"], row["user_input"], qa_prompt_template)
    finals.append(final)
    raws.append(raw)
over_cautious_df["answer_reasoning"] = finals
over_cautious_df["raw_reasoning"] = raws

refused = over_cautious_df["answer_reasoning"].str.lower().str.contains("not present")
print(f"Récupérées : {(~refused).sum()}/{len(over_cautious_df)}")
print("Sorties sans 'Answer:' :", (~over_cautious_df["raw_reasoning"].str.contains("Answer:")).sum())
print(over_cautious_df.loc[~refused, "category"].value_counts().to_string())


100%|██████████| 51/51 [13:38<00:00, 16.04s/it]

Récupérées : 24/51
Sorties sans 'Answer:' : 0
category
Renewal Term                          3
Expiration Date                       2
Change Of Control                     2
Unlimited/All-You-Can-Eat-License     2
Agreement Date                        2
Termination For Convenience           2
Parties                               1
Irrevocable Or Perpetual License      1
Source Code Escrow                    1
Non-Transferable License              1
Notice Period To Terminate Renewal    1
Non-Compete                           1
Competitive Restriction Exception     1
Uncapped Liability                    1
Audit Rights                          1
Warranty Duration                     1
Affiliate License-Licensor            1


In [ ]:
def overlaps_reference(answer, reference):
    a = normalize_text(answer)
    r = normalize_text(reference)
    return r[:60] in a or a[:60] in r

recovered_df = over_cautious_df[~refused].copy()
recovered_df["quotes_reference"] = [
    overlaps_reference(a, r) for a, r in zip(recovered_df["answer_reasoning"], recovered_df["reference"])
]
print(f"Réponses récupérées qui citent la référence : {recovered_df['quotes_reference'].sum()}/{len(recovered_df)}")


Réponses récupérées qui citent la référence : 17/24


In [ ]:
new_negative_r = []
for item in tqdm(negative_results_v3_qwen):
    final, raw = regenerate_with_reasoning(item["contexts"], item["question"], qa_prompt_template)
    new_negative_r.append({**item, "answer": final, "raw": raw, "classification": classify_negative(final)})

counts = Counter(x["classification"] for x in new_negative_r)
print(counts)
print(f"Taux de FP raisonnement + prompt v1 : {counts['false positive'] / len(new_negative_r):.1%}")


100%|██████████| 118/118 [24:55<00:00, 12.67s/it]

Counter({'true negative': 107, 'false positive': 11})
Taux de FP raisonnement + prompt v1 : 9.3%


In [30]:
with open("../artifacts/baseline_v3/negative_results_v3_qwen.json") as f:
    negative_results_v3_qwen = json.load(f)

new_negative_r2 = []
for item in tqdm(negative_results_v3_qwen):
    final, raw = regenerate_with_reasoning(item["contexts"], item["question"], qa_prompt_template_v2)
    new_negative_r2.append({**item, "answer": final, "raw": raw, "classification": classify_negative(final)})

counts = Counter(x["classification"] for x in new_negative_r2)
print(counts)
print(f"Taux de FP prompt v2 + raisonnement : {counts['false positive'] / len(new_negative_r2):.1%}")

with open("../artifacts/baseline_v3/negative_results_v3_qwen_reasoning_v2.json", "w") as f:
    json.dump(new_negative_r2, f, indent=2)


100%|██████████| 118/118 [1:00:55<00:00, 30.98s/it]

Counter({'true negative': 93, 'false positive': 25})
Taux de FP prompt v2 + raisonnement : 21.2%


In [32]:
def run_generation_reasoning_t(old_results, output_path, template):
    if os.path.exists(output_path):
        with open(output_path) as f:
            results = json.load(f)
        done = {(r["title"], r["question"]) for r in results}
    else:
        results, done = [], set()

    remaining = [r for r in old_results if (r["title"], r["question"]) not in done]
    print(f"Reprise : {len(results)} faits, {len(remaining)} restants")

    for item in tqdm(remaining):
        try:
            final, raw = regenerate_with_reasoning(item["contexts"], item["question"], template)
        except Exception as e:
            final, raw = f"[ERROR: {type(e).__name__}: {e}]", ""
        results.append({**item, "answer": final, "raw_answer": raw})
        with open(output_path, "w") as f:
            json.dump(results, f, indent=2)
    return results

with open("../artifacts/baseline_v3/positive_results_v3_qwen.json") as f:
    positive_results_v3_qwen = json.load(f)

positive_results_v3_qwen_reasoning_v2 = run_generation_reasoning_t(
    positive_results_v3_qwen,
    "../artifacts/baseline_v3/positive_results_v3_qwen_reasoning_v2.json",
    qa_prompt_template_v2,
)


Reprise : 77 faits, 128 restants


100%|██████████| 128/128 [34:54<00:00, 16.37s/it]


In [33]:
new_by_key = {(r["title"], r["question"]): r for r in positive_results_v3_qwen_reasoning_v2}
recovered_v2r = sum(
    "not present" not in new_by_key[(row["title"], row["user_input"])]["answer"].lower()
    for _, row in over_cautious_df.iterrows()
)
print(f"Abstentions injustifiées récupérées (prompt v2 + raisonnement) : {recovered_v2r}/{len(over_cautious_df)} (v1 + raisonnement : 24/51)")


Abstentions injustifiées récupérées (prompt v2 + raisonnement) : 22/51 (v1 + raisonnement : 24/51)


In [34]:
n_np = sum("not present" in r["answer"].lower() for r in positive_results_v3_qwen_reasoning_v2)
print(f"'not present' sur les positifs : {n_np}/205 (v1 + raisonnement : 72/205)")

df_ragas_reasoning_v2, summary_global_reasoning_v2, summary_by_cat_reasoning_v2 = evaluate_generation(
    positive_results_v3_qwen_reasoning_v2, "v3_qwen_reasoning_v2"
)
summary_global_reasoning_v2


'not present' sur les positifs : 74/205 (v1 + raisonnement : 72/205)
205 usable after removing errors (0 errors)
batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [01:27<00:00,  1.14it/s]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [01:47<00:00,  1.07s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [02:51<00:00,  1.72s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 25/25 [00:36<00:00,  1.47s/it]


faithfulness          0.694028
answer_relevancy      0.647696
context_precision     0.626992
context_recall        0.707437
answer_correctness    0.463541
dtype: float64

## Full run of the reasoning variant

The subset tests cannot show side effects on examples that were already answered correctly, so the whole positive sample is regenerated with the reasoning step:
- the 118 negative answers computed above are saved as they are;
- the 205 positive examples are regenerated from the stored chunks (`run_generation_reasoning`, resumable). This needs neither retrieval nor the embedding models, only the LLM and the prompt, so it is much lighter on memory;
- the answers are then scored with the same ragas evaluation as before.


In [ ]:
with open("../artifacts/baseline_v3/negative_results_v3_qwen_reasoning.json", "w") as f:
    json.dump(new_negative_r, f, indent=2)


In [ ]:
import os

def run_generation_reasoning(old_results, output_path):
    if os.path.exists(output_path):
        with open(output_path) as f:
            results = json.load(f)
        done = {(r["title"], r["question"]) for r in results}
    else:
        results, done = [], set()

    remaining = [r for r in old_results if (r["title"], r["question"]) not in done]
    print(f"Reprise : {len(results)} faits, {len(remaining)} restants")

    for item in tqdm(remaining):
        try:
            final, raw = regenerate_with_reasoning(item["contexts"], item["question"], qa_prompt_template)
        except Exception as e:
            final, raw = f"[ERROR: {type(e).__name__}: {e}]", ""
        results.append({**item, "answer": final, "raw_answer": raw})
        with open(output_path, "w") as f:
            json.dump(results, f, indent=2)
    return results

with open("../artifacts/baseline_v3/positive_results_v3_qwen.json") as f:
    positive_results_v3_qwen = json.load(f)

positive_results_v3_qwen_reasoning = run_generation_reasoning(
    positive_results_v3_qwen,
    "../artifacts/baseline_v3/positive_results_v3_qwen_reasoning.json",
)


Reprise : 0 faits, 205 restants


100%|██████████| 205/205 [52:29<00:00, 15.36s/it]


In [ ]:
df_ragas_reasoning, summary_global_reasoning, summary_by_cat_reasoning = evaluate_generation(
    positive_results_v3_qwen_reasoning, "v3_qwen_reasoning"
)
summary_global_reasoning


205 usable after removing errors (0 errors)
batch 1/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [01:17<00:00,  1.29it/s]


batch 2/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [01:48<00:00,  1.08s/it]


batch 3/11 (20 examples)...


...

batch 10/11 (20 examples)...


Evaluating: 100%|██████████| 100/100 [03:27<00:00,  2.08s/it]


batch 11/11 (5 examples)...


Evaluating: 100%|██████████| 25/25 [00:36<00:00,  1.46s/it]


faithfulness          0.697779
answer_relevancy      0.650820
context_precision     0.633963
context_recall        0.710484
answer_correctness    0.479475
dtype: float64

## Results: qwen2.5:7b with a reasoning step

| Metric | v1 | v3 + llama3.2:3b | v3 + qwen (prompt v1) | **v3 + qwen + reasoning** |
|---|---|---|---|---|
| faithfulness | 0.757 | 0.828 | 0.573 | 0.698 |
| answer_relevancy | 0.614 | 0.624 | 0.654 | 0.651 |
| context_precision | 0.409 | 0.588 | 0.625 | 0.634 |
| context_recall | 0.477 | 0.676 | 0.708 | 0.710 |
| answer_correctness | 0.411 | 0.489 | 0.403 | **0.479** |
| "not present" on positives | – | 13% | 49% | 35% |
| False positives on negatives | 64.4% | 71.2% | 3.4% | **9.3%** |

- `answer_correctness` is back to the level of llama (0.479 vs 0.489, within the judge noise), while false positives drop from 71.2% to 9.3%.
- The `faithfulness` gap with llama is mostly the abstention artifact: the cell below shows that answered examples reach 0.897, abstentions only 0.33.


In [ ]:
df_r = df_ragas_reasoning.copy()
df_r["is_not_present"] = df_r["response"].str.lower().str.contains("not present")
df_r.groupby("is_not_present")[["faithfulness", "answer_correctness"]].mean().round(3)


,faithfulness,answer_correctness
is_not_present,,
False,0.897,0.665
True,0.330,0.142


## Final baseline v3 configuration

- **Chunking:** fixed, 512 tokens
- **Retrieval:** bge-large embeddings, hybrid dense + BM25, scoped to one contract
- **Reranking:** `bge-reranker-v2-m3` with `strip_boilerplate`
- **Tier 1:** first chunk for `Document Name` and `Parties`
- **Tier 3:** widened pool (15 → 5) on 10 categories
- **Generation:** `qwen2.5:7b`, prompt v1 with a reasoning step (only the text after `Answer:` is kept)

Compared with baseline v1: `answer_correctness` 0.411 → 0.479, false positives 64.4% → 9.3%, `context_precision` 0.409 → 0.634, `context_recall` 0.477 → 0.710.

llama3.2:3b is slightly better on positive examples alone (`answer_correctness` 0.489 vs 0.479, within the judge noise), but it invents a clause in 71% of the negative examples. qwen2.5:7b was chosen because it is much more reliable on that side.


## Open items

- **Prompt v2 + reasoning:** tested on both samples and rejected — 21% false positives (vs 9.3% for reasoning alone) with no gain on `answer_correctness` or on the abstention rate.
- **Remaining abstentions (72 of 205 positives):** many are justified by retrieval failures (`Minimum Commitment`, `Volume Restriction`, `Most Favored Nation`, ...), where the right chunk is missing or ranked too low.
- **Legal-domain embedding or reranker:** a model such as `freelawproject/modernbert-embed-base_finetune_512` was identified but never tested.
- **Confidence threshold on reranker scores:** not needed for now, since the model change handles the false positives.
